In [2]:
from pyspark.sql import SparkSession
import pandas as pd

# Create Spark session
spark = SparkSession.builder.appName("CreditRiskAnalysis").getOrCreate()

# Load data
df = spark.read.csv("credit_risk_dataset.csv", header=True, inferSchema=True)
df.show(5)

# Load also with pandas
pdf = pd.read_csv("credit_risk_dataset.csv")
pdf.head()


+----------+-------------+---------------------+-----------------+-----------+----------+---------+-------------+-----------+-------------------+-------------------------+--------------------------+
|person_age|person_income|person_home_ownership|person_emp_length|loan_intent|loan_grade|loan_amnt|loan_int_rate|loan_status|loan_percent_income|cb_person_default_on_file|cb_person_cred_hist_length|
+----------+-------------+---------------------+-----------------+-----------+----------+---------+-------------+-----------+-------------------+-------------------------+--------------------------+
|        22|        59000|                 RENT|            123.0|   PERSONAL|         D|    35000|        16.02|          1|               0.59|                        Y|                         3|
|        21|         9600|                  OWN|              5.0|  EDUCATION|         B|     1000|        11.14|          0|                0.1|                        N|                         2|
|    

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


Calculer le montant total et la durée moyenne des crédits non remboursés

In [4]:

df.createOrReplaceTempView("credits")
df.printSchema()
df.columns



root
 |-- person_age: integer (nullable = true)
 |-- person_income: integer (nullable = true)
 |-- person_home_ownership: string (nullable = true)
 |-- person_emp_length: double (nullable = true)
 |-- loan_intent: string (nullable = true)
 |-- loan_grade: string (nullable = true)
 |-- loan_amnt: integer (nullable = true)
 |-- loan_int_rate: double (nullable = true)
 |-- loan_status: integer (nullable = true)
 |-- loan_percent_income: double (nullable = true)
 |-- cb_person_default_on_file: string (nullable = true)
 |-- cb_person_cred_hist_length: integer (nullable = true)



['person_age',
 'person_income',
 'person_home_ownership',
 'person_emp_length',
 'loan_intent',
 'loan_grade',
 'loan_amnt',
 'loan_int_rate',
 'loan_status',
 'loan_percent_income',
 'cb_person_default_on_file',
 'cb_person_cred_hist_length']

Calculer le montant total et la durée moyenne des crédits non remboursés.


In [5]:
spark.sql("""
SELECT 
    SUM(loan_amnt) AS total_amount,
    AVG(person_emp_length) AS avg_employment_length
FROM credits
WHERE loan_status = 1
""").show()

+------------+---------------------+
|total_amount|avg_employment_length|
+------------+---------------------+
|    77125375|    4.137562261939642|
+------------+---------------------+



Calculer le pourcentage des crédits remboursés et non remboursés.

In [6]:
spark.sql("""
SELECT 
    CASE WHEN loan_status = 1 THEN 'Non rembourse' ELSE 'Rembourse' END as statut,
    COUNT(*) * 100.0 / (SELECT COUNT(*) FROM credits) AS percentage
FROM credits
GROUP BY loan_status
""").show()

+-------------+-----------------+
|       statut|       percentage|
+-------------+-----------------+
|Non rembourse|21.81639605905282|
|    Rembourse|78.18360394094718|
+-------------+-----------------+



Quelle est la distribution des différents états de propriété de la maison

In [7]:
spark.sql("""
select 
    sum(loan_status) as total_non_rembourse,
    count(*) * 100 as total_count,
    sum(case when loan_status = 0 then 1 else 0 end) * 100.0 / count(*) as pers_rembourse,
    sum(case when loan_status = 1 then 1 else 0 end) * 100.0 / count(*) as pers_non_remboursee
from credits
""").show()

+-------------------+-----------+-----------------+-------------------+
|total_non_rembourse|total_count|   pers_rembourse|pers_non_remboursee|
+-------------------+-----------+-----------------+-------------------+
|               7108|    3258100|78.18360394094718|  21.81639605905282|
+-------------------+-----------+-----------------+-------------------+



In [6]:
spark.sql("""
SELECT person_home_ownership, COUNT(*) AS total_types
FROM credits
GROUP BY person_home_ownership
""").show()


+---------------------+-----------+
|person_home_ownership|total_types|
+---------------------+-----------+
|                  OWN|       2584|
|                 RENT|      16446|
|             MORTGAGE|      13444|
|                OTHER|        107|
+---------------------+-----------+



Quels sont les 3 motifs de crédits les plus demandés

In [7]:
spark.sql(""" select loan_intent, count(*) as loan 
from credits
GROUP BY loan_intent
limit(3)
""").show()


+-----------------+----+
|      loan_intent|loan|
+-----------------+----+
|DEBTCONSOLIDATION|5212|
|          VENTURE|5719|
|         PERSONAL|5521|
+-----------------+----+



Solution avec Api DataFrame

In [9]:
df.agg({"loan_amnt": "sum", "person_emp_length": "avg"})\
  .withColumnRenamed("avg(person_emp_length)", "duree_moyen")\
  .withColumnRenamed("sum(loan_amnt)", "montant")\
  .show()

+-----------------+---------+
|      duree_moyen|  montant|
+-----------------+---------+
|4.789686296787225|312431300|
+-----------------+---------+



Solution avec Pandas

In [12]:
import pyspark.pandas as ps 
df2 = ps.DataFrame(df)
df2.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [16]:
df2.agg({"loan_amnt": "sum", "person_emp_length": "avg"}).head()
  

loan_amnt            3.124313e+08
person_emp_length    4.789686e+00
dtype: float64

Calculer le pourcentage des crédits remboursés et non remboursés par motif puis par grade.

In [23]:
spark.sql("""
SELECT 
    loan_intent ,
    loan_grade,
    loan_status,
    COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY loan_intent, loan_grade) AS percentage
FROM credits
GROUP BY loan_intent, loan_grade, loan_status
ORDER BY loan_intent, loan_grade, percentage DESC
limit 10
""").show()

+-----------------+----------+-----------+------------------+
|      loan_intent|loan_grade|loan_status|        percentage|
+-----------------+----------+-----------+------------------+
|DEBTCONSOLIDATION|         A|          0| 88.76212207644039|
|DEBTCONSOLIDATION|         A|          1| 11.23787792355961|
|DEBTCONSOLIDATION|         B|          0| 81.29629629629630|
|DEBTCONSOLIDATION|         B|          1| 18.70370370370370|
|DEBTCONSOLIDATION|         C|          0| 77.25527831094050|
|DEBTCONSOLIDATION|         C|          1| 22.74472168905950|
|DEBTCONSOLIDATION|         D|          1| 92.66666666666667|
|DEBTCONSOLIDATION|         D|          0|  7.33333333333333|
|DEBTCONSOLIDATION|         E|          1|100.00000000000000|
|DEBTCONSOLIDATION|         F|          1|100.00000000000000|
+-----------------+----------+-----------+------------------+



In [28]:
spark.sql("""
SELECT 
    loan_intent ,
    loan_status,
    count(*) as nb
FROM credits
GROUP BY loan_intent, loan_status
ORDER BY 1
""").show()

+-----------------+-----------+--------+
|      loan_intent|loan_status|count(1)|
+-----------------+-----------+--------+
|DEBTCONSOLIDATION|          1|    1490|
|DEBTCONSOLIDATION|          0|    3722|
|        EDUCATION|          1|    1111|
|        EDUCATION|          0|    5342|
|  HOMEIMPROVEMENT|          0|    2664|
|  HOMEIMPROVEMENT|          1|     941|
|          MEDICAL|          1|    1621|
|          MEDICAL|          0|    4450|
|         PERSONAL|          0|    4423|
|         PERSONAL|          1|    1098|
|          VENTURE|          0|    4872|
|          VENTURE|          1|     847|
+-----------------+-----------+--------+



In [32]:
spark.sql("""
SELECT 
    loan_intent,
    loan_status,
    COUNT(*) as nbc
FROM credits
GROUP BY loan_intent, loan_status
""").show()

+-----------------+-----------+----+
|      loan_intent|loan_status| nbc|
+-----------------+-----------+----+
|DEBTCONSOLIDATION|          1|1490|
|          MEDICAL|          1|1621|
|  HOMEIMPROVEMENT|          0|2664|
|         PERSONAL|          0|4423|
|        EDUCATION|          1|1111|
|  HOMEIMPROVEMENT|          1| 941|
|        EDUCATION|          0|5342|
|          MEDICAL|          0|4450|
|DEBTCONSOLIDATION|          0|3722|
|          VENTURE|          0|4872|
|         PERSONAL|          1|1098|
|          VENTURE|          1| 847|
+-----------------+-----------+----+



In [31]:
spark.sql("""
SELECT 
    t1.loan_intent,
    t1.loan_status,
    t1.nbc * 100.0 / t2.nb as pourcentage 
FROM (
    SELECT 
        loan_intent,
        loan_status,
        COUNT(*) as nbc
    FROM credits
    GROUP BY loan_intent, loan_status
) t1,
(
    SELECT 
        loan_intent,
        COUNT(*) as nb
    FROM credits
    GROUP BY loan_intent
) t2
WHERE t1.loan_intent = t2.loan_intent
""").show()

+-----------------+-----------+-----------------+
|      loan_intent|loan_status|      pourcentage|
+-----------------+-----------+-----------------+
|DEBTCONSOLIDATION|          0|71.41212586339217|
|DEBTCONSOLIDATION|          1|28.58787413660783|
|          VENTURE|          1|14.81028151774786|
|          VENTURE|          0|85.18971848225214|
|         PERSONAL|          1|19.88770150335084|
|         PERSONAL|          0|80.11229849664916|
|        EDUCATION|          0|82.78320161165349|
|        EDUCATION|          1|17.21679838834651|
|  HOMEIMPROVEMENT|          1|26.10263522884882|
|  HOMEIMPROVEMENT|          0|73.89736477115118|
|          MEDICAL|          0|73.29929171470927|
|          MEDICAL|          1|26.70070828529073|
+-----------------+-----------+-----------------+

